# Tugas Modul Sesi 2 & 3 Minggu 12 — RNN dan LSTM
**Mata Kuliah:** Kecerdasan Buatan

**Nama** : Oktova Yoga Praditia Samosir 

**NIM**  : 42324001

**Tujuan:** Memahami konsep dan mengimplementasikan algoritma **Recurrent Neural Network (RNN)** dan **Long Short-Term Memory (LSTM)** untuk memprediksi data *time series* penjualan ritel bulanan (dataset `RSCCASN.csv`).

## Ringkasan Teori

**Time Series** adalah data yang diukur pada interval waktu teratur, sehingga urutan waktunya penting. Karakteristiknya: *trend* (pergerakan jangka panjang), *seasonality* (pola berulang dalam periode tetap), *cyclical* (fluktuasi jangka panjang tidak tetap), dan *random/noise*.

**RNN** adalah neural network untuk data sekuensial. RNN memproses elemen urutan satu per satu dan menyimpan memori langkah sebelumnya di dalam **hidden state**. Bobot **U** menghubungkan input ke hidden state, **W** menghubungkan hidden state antar *time step*, dan **V** menghubungkan hidden state ke output. Kelemahannya adalah *vanishing* dan *exploding gradient*, sehingga RNN sulit mempelajari ketergantungan jangka panjang (*long-term dependencies*).

**LSTM** (Hochreiter & Schmidhuber, 1997) adalah varian RNN yang mengatasi *vanishing gradient* dengan **cell state** sebagai memori jangka panjang yang diperbarui secara *aditif*, serta tiga gerbang (gate):
- **Forget gate** ($f_t$): menentukan informasi dari $C_{t-1}$ yang dibuang.
- **Input gate** ($i_t$) dan kandidat $\tilde{C}_t$: menentukan informasi baru yang ditambahkan.
- **Cell state**: $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
- **Output gate** ($o_t$): menghasilkan hidden state $h_t = o_t * \tanh(C_t)$.

Pada tugas ini, kedua model digunakan dengan tipe **many-to-one**: 12 bulan sebelumnya dipakai sebagai input untuk memprediksi 1 bulan berikutnya.

## 1. Persiapan Lingkungan Pengembangan
Notebook ini dijalankan di **Google Colab** karena TensorFlow, NumPy, Pandas, dan Matplotlib sudah tersedia di sana. Perintah di bawah memastikan TensorFlow terpasang (di Colab biasanya sudah ada).

In [1]:
!pip install -q tensorflow

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\Oktovaaaaa\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


### Import Library
- `numpy`, `pandas`: manipulasi data numerik dan tabel.
- `matplotlib`, `seaborn`: visualisasi data.
- `tensorflow`/`keras`: membangun model `SimpleRNN` dan `LSTM`.
- `MinMaxScaler`: normalisasi data ke rentang 0–1.
- `TimeseriesGenerator`: membentuk pasangan input–target dari data sekuensial.
- `EarlyStopping`: menghentikan pelatihan jika `val_loss` tidak membaik.

Pada TensorFlow versi baru (Keras 3), `TimeseriesGenerator` sudah dihapus. Karena itu, jika impor gagal, dibuat kelas pengganti dengan perilaku yang sama: input `length` langkah, target pada langkah berikutnya. Seed juga diatur agar hasil lebih konsisten saat dijalankan ulang.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.dates as mdates

try:
    from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
except ImportError:
    # Pengganti TimeseriesGenerator untuk Keras 3 (perilaku sama dengan versi lama)
    class TimeseriesGenerator(tf.keras.utils.PyDataset):
        def __init__(self, data, targets, length, batch_size=1, **kwargs):
            super().__init__(**kwargs)
            self.data = np.asarray(data, dtype='float32')
            self.targets = np.asarray(targets, dtype='float32')
            self.length = length
            self.batch_size = batch_size
            self.n = len(self.data) - length
        def __len__(self):
            return int(np.ceil(self.n / self.batch_size))
        def __getitem__(self, idx):
            rows = range(idx * self.batch_size, min((idx + 1) * self.batch_size, self.n))
            X = np.array([self.data[i:i + self.length] for i in rows])
            y = np.array([self.targets[i + self.length] for i in rows])
            return X, y

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow version:', tf.__version__)

## 2. Unduh dan Muat Dataset
Dataset **Retail Sales** (`RSCCASN.csv`) berisi penjualan ritel bulanan dari Januari 1992 sampai Oktober 2019 (334 baris). Sel di bawah meminta Anda mengunggah file jika `RSCCASN.csv` belum ada di direktori kerja Colab.

In [ ]:
import os
if not os.path.exists('RSCCASN.csv'):
    from google.colab import files
    uploaded = files.upload()  # pilih file RSCCASN.csv

## 3. Pengolahan Data (Data Processing)
- Dataset dibaca dengan kolom `DATE` sebagai index bertipe tanggal (`parse_dates=True`).
- Nama kolom `RSCCASN` diubah menjadi `Sales`.
- Data ditampilkan secara singkat (`head`, `info`, `describe`) untuk mengenali strukturnya.

In [ ]:
# Membaca dataset
dataset_path = 'RSCCASN.csv'
sales_data = pd.read_csv(dataset_path, parse_dates=True, index_col='DATE')
sales_data.columns = ['Sales']

print(sales_data.head())
print()
sales_data.info()
sales_data.describe()

### Visualisasi Data Penjualan
Plot di bawah memperlihatkan karakteristik *time series* dari materi:
- **Trend** naik dalam jangka panjang.
- **Seasonality** yang kuat, dengan puncak setiap **Desember** (musim belanja akhir tahun) dan penurunan pada Januari.
- Penurunan sekitar 2008–2009 yang mencerminkan pola **cyclical** (krisis ekonomi).

In [ ]:
# Plot data penjualan
sales_data.plot(figsize=(12, 8))
plt.title('Retail Sales (RSCCASN) per Bulan', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True)
plt.show()

### Pembagian Dataset (Train-Test Split)
Pada time series, data **tidak diacak**. Data terakhir dipakai sebagai data uji agar model diuji pada masa "depan" yang belum dilihatnya. Di sini **18 bulan terakhir** menjadi data *test* dan sisanya menjadi data *train*.

In [ ]:
# Train-test split
test_size = 18
test_index = len(sales_data) - test_size
train = sales_data.iloc[:test_index]
test = sales_data.iloc[test_index:].copy()  # .copy() agar aman saat menambah kolom prediksi

print('Jumlah data train:', len(train), '| periode', train.index[0].date(), 's/d', train.index[-1].date())
print('Jumlah data test :', len(test), '| periode', test.index[0].date(), 's/d', test.index[-1].date())

## 4. Normalisasi Data
- `MinMaxScaler()` menskalakan data ke rentang 0–1 agar pelatihan neural network lebih stabil.
- `scaler.fit(train)` menghitung nilai minimum dan maksimum **hanya dari data latih** untuk mencegah *data leakage*.
- `scaler.transform` mengubah data train dan test dengan skala tersebut.

In [ ]:
# Inisialisasi MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(train)

# Transformasi data train dan test
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)

print('5 data train pertama (skala 0-1):')
print(scaled_train[:5].ravel())

## 5. Membuat Time Series Generator
- **Panjang sekuens** `length = 12`: model melihat 12 bulan sebelumnya (satu siklus musiman penuh) untuk memprediksi bulan berikutnya.
- `n_features = 1`: hanya ada satu variabel, yaitu `Sales`.
- `TimeseriesGenerator` membentuk pasangan input $X$ (12 nilai berurutan) dan target $y$ (nilai ke-13), dengan `batch_size=1`.

In [ ]:
# Membuat generator data time series
length = 12  # Panjang input (12 bulan)
n_features = 1
generator = TimeseriesGenerator(scaled_train, scaled_train, length=length, batch_size=1)

print('Jumlah sampel pada generator:', len(generator))
X, y = generator[0]
print('Bentuk input X :', X.shape)
print('Input X pertama :', X.ravel())
print('Target y pertama:', y.ravel())

## 6. Membangun Model RNN
- **Arsitektur:** satu layer `SimpleRNN` dengan 100 unit dan aktivasi ReLU, diikuti `Dense(1)` sebagai output prediksi satu nilai (many-to-one).
- **Kompilasi:** optimizer `adam` dan loss `mse` (*mean squared error*).
- **Early Stopping:** pelatihan dihentikan jika `val_loss` tidak membaik selama 3 epoch, lalu bobot terbaik dipulihkan.
- **Validation generator:** dibuat dari data test (18 bulan) sehingga menghasilkan 18 − 12 = 6 sampel validasi.
- **Pelatihan:** maksimal 20 epoch.

In [ ]:
# Model SimpleRNN
model_rnn = Sequential()
model_rnn.add(SimpleRNN(100, activation='relu', input_shape=(length, n_features)))
model_rnn.add(Dense(1))
model_rnn.compile(optimizer='adam', loss='mse')
model_rnn.summary()

Jumlah parameter `SimpleRNN`: $100 \times (1 + 100 + 1) = 10{,}200$, yaitu bobot **U** (input→hidden), **W** (hidden→hidden), dan bias. Layer `Dense` (bobot **V**) memiliki $100 + 1 = 101$ parameter.

In [ ]:
# Callback early stopping untuk RNN
early_stop_rnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Validation generator
validation_generator = TimeseriesGenerator(scaled_test, scaled_test, length=length, batch_size=1)

# Melatih model RNN
history_rnn = model_rnn.fit(generator, epochs=20,
                            validation_data=validation_generator,
                            callbacks=[early_stop_rnn])

## 7. Membangun Model LSTM
- **Arsitektur:** layer `LSTM` dengan 100 unit (aktivasi ReLU), lalu `Dropout(0.2)` yang menonaktifkan 20% neuron secara acak saat pelatihan untuk mengurangi *overfitting*, dan `Dense(1)` sebagai output.
- **Kompilasi:** optimizer Adam dengan *learning rate* 0.001 dan loss `mse`.
- **Early Stopping:** `patience=5`, lebih longgar karena LSTM biasanya konvergen lebih lambat.
- **Pelatihan:** maksimal 20 epoch dengan data validasi yang sama.

In [ ]:
# Model LSTM dengan penambahan Dropout
model_lstm = Sequential()
model_lstm.add(LSTM(100, activation='relu', input_shape=(length, n_features)))
model_lstm.add(Dropout(0.2))  # Penambahan Dropout untuk mengurangi overfitting
model_lstm.add(Dense(1))
model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model_lstm.summary()

Jumlah parameter `LSTM` adalah $4 \times 100 \times (1 + 100 + 1) = 40{,}800$, sekitar 4 kali `SimpleRNN`. Ini karena LSTM memiliki empat set bobot: *forget gate*, *input gate*, kandidat cell state, dan *output gate*. Hal ini sesuai dengan materi bahwa LSTM memiliki kompleksitas komputasi lebih tinggi.

In [ ]:
# Callback early stopping untuk LSTM
early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Melatih model LSTM
history_lstm = model_lstm.fit(generator, epochs=20,
                              validation_data=validation_generator,
                              callbacks=[early_stop_lstm])

## 8. Evaluasi Model: Kurva Loss
- Riwayat `loss` dan `val_loss` dari `history` disimpan dalam DataFrame.
- Fungsi `plot_loss` menampilkan kurva kedua model dalam satu grafik. Fungsi `plot_loss_rnn` dan `plot_loss_lstm` menampilkan masing-masing model secara terpisah.
- Kurva ini dipakai untuk memantau **overfitting** (loss train turun tetapi val_loss naik) atau **underfitting** (keduanya tetap tinggi).

In [ ]:
# Fungsi untuk plot training dan validation loss
def plot_loss(history_rnn, history_lstm):
    losses_rnn = pd.DataFrame(history_rnn.history)
    losses_lstm = pd.DataFrame(history_lstm.history)

    plt.figure(figsize=(10, 6))
    plt.plot(losses_rnn['loss'], label='RNN Loss', linewidth=2)
    plt.plot(losses_rnn['val_loss'], label='RNN Val Loss', linewidth=2)
    plt.plot(losses_lstm['loss'], label='LSTM Loss', linewidth=2)
    plt.plot(losses_lstm['val_loss'], label='LSTM Val Loss', linewidth=2)
    plt.title('Training and Validation Loss', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

plot_loss(history_rnn, history_lstm)

In [ ]:
def plot_loss_rnn(history_rnn):
    losses_rnn = pd.DataFrame(history_rnn.history)
    plt.figure(figsize=(10, 6))
    plt.plot(losses_rnn['loss'], label='RNN Loss', linewidth=2)
    plt.plot(losses_rnn['val_loss'], label='RNN Val Loss', linewidth=2)
    plt.title('Training and Validation Loss - RNN', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

def plot_loss_lstm(history_lstm):
    losses_lstm = pd.DataFrame(history_lstm.history)
    plt.figure(figsize=(10, 6))
    plt.plot(losses_lstm['loss'], label='LSTM Loss', linewidth=2)
    plt.plot(losses_lstm['val_loss'], label='LSTM Val Loss', linewidth=2)
    plt.title('Training and Validation Loss - LSTM', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

plot_loss_rnn(history_rnn)
plot_loss_lstm(history_lstm)

## 9. Prediksi Model pada Data Test
Fungsi `make_predictions` melakukan **prediksi rekursif** (*recursive multi-step forecasting*):
1. Ambil 12 data terakhir dari data train sebagai batch awal.
2. Prediksi 1 bulan berikutnya.
3. Geser jendela: buang nilai paling lama, tambahkan nilai prediksi ke akhir.
4. Ulangi sebanyak panjang data test (18 bulan).

Hasil prediksi dikembalikan ke skala asli dengan `scaler.inverse_transform`, lalu ditambahkan sebagai kolom baru pada DataFrame `test`.

In [ ]:
# Membuat prediksi untuk data test dengan RNN dan LSTM
def make_predictions(model, scaled_data, length, n_features):
    predictions = []
    first_eval_batch = scaled_data[-length:]
    current_batch = first_eval_batch.reshape((1, length, n_features))

    for i in range(len(test)):
        current_pred = model.predict(current_batch, verbose=0)[0]
        predictions.append(current_pred)
        current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

    return scaler.inverse_transform(predictions)

rnn_predictions = make_predictions(model_rnn, scaled_train, length, n_features)
lstm_predictions = make_predictions(model_lstm, scaled_train, length, n_features)

# Menambahkan prediksi ke dataframe test
test['RNN_Predictions'] = rnn_predictions
test['LSTM_Predictions'] = lstm_predictions
test.round(0)

### Metrik Evaluasi
Sebagai pelengkap visualisasi, akurasi kedua model dihitung secara kuantitatif dengan **MAE** (rata-rata selisih absolut), **RMSE** (akar rata-rata kuadrat error), dan **MAPE** (persentase error rata-rata). Semakin kecil nilainya, semakin baik modelnya.

In [ ]:
def evaluate(actual, pred):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    return mae, rmse, mape

metrics = pd.DataFrame(
    [evaluate(test['Sales'], test['RNN_Predictions']),
     evaluate(test['Sales'], test['LSTM_Predictions'])],
    index=['RNN', 'LSTM'], columns=['MAE', 'RMSE', 'MAPE (%)'])
metrics.round(2)

### Plot Prediksi vs Data Aktual
Fungsi `plot_predictions` menampilkan penjualan aktual (biru) bersama prediksi RNN (oranye, putus-putus) dan LSTM (hijau, putus-putus) pada periode test. Sumbu x diformat per 3 bulan agar tanggal mudah dibaca.

In [ ]:
# Fungsi untuk plot prediksi RNN dan LSTM bersama data actual
def plot_predictions(test, sales_data, rnn_predictions, lstm_predictions):
    plt.figure(figsize=(14, 8))
    plt.plot(test.index, test['Sales'], label='Actual Sales', color='blue', linewidth=2)
    plt.plot(test.index, test['RNN_Predictions'], label='RNN Predictions', color='orange', linestyle='--', linewidth=2)
    plt.plot(test.index, test['LSTM_Predictions'], label='LSTM Predictions', color='green', linestyle='--', linewidth=2)

    # Formatting the x-axis to show dates nicely
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.gcf().autofmt_xdate()  # Rotate the x labels for better readability

    plt.title('Actual vs RNN and LSTM Predictions', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.xlim([test.index[0], test.index[-1]])
    plt.show()

plot_predictions(test, sales_data, rnn_predictions, lstm_predictions)

## 10. Prediksi Jangka Panjang (12 Bulan ke Depan)
Fungsi `forecast_future` bekerja dengan cara rekursif yang sama, tetapi untuk `periods = 12` bulan **setelah data terakhir** (Oktober 2019), yaitu November 2019 sampai Oktober 2020.

**Catatan:** Kode modul memakai `scaled_train` sebagai titik awal. Akibatnya, forecast dimulai dari akhir data train (April 2018), padahal diberi label mulai November 2019. Agar forecast benar-benar melanjutkan data terbaru, notebook ini memakai **seluruh data** yang sudah dinormalisasi (`scaled_full`) sebagai batch awal. Model dan scaler tetap sama.

In [ ]:
# Seluruh data dinormalisasi dengan scaler yang sama (fit pada data train)
scaled_full = scaler.transform(sales_data[['Sales']])

# Membuat prediksi ke depan dengan model
def forecast_future(model, scaled_data, length, n_features, periods):
    forecast = []
    first_eval_batch = scaled_data[-length:]
    current_batch = first_eval_batch.reshape((1, length, n_features))

    for i in range(periods):
        current_pred = model.predict(current_batch, verbose=0)[0]
        forecast.append(current_pred)
        current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

    return scaler.inverse_transform(forecast)

# Prediksi untuk 12 periode ke depan
forecast_rnn = forecast_future(model_rnn, scaled_full, length, n_features, periods=12)
forecast_lstm = forecast_future(model_lstm, scaled_full, length, n_features, periods=12)

# Membuat dataframe untuk forecast
forecast_index = pd.date_range(start='2019-11-01', periods=12, freq='MS')
forecast_rnn_data = pd.DataFrame(data=forecast_rnn, index=forecast_index, columns=['RNN_Forecast'])
forecast_lstm_data = pd.DataFrame(data=forecast_lstm, index=forecast_index, columns=['LSTM_Forecast'])

pd.concat([forecast_rnn_data, forecast_lstm_data], axis=1).round(0)

### Plot Forecast
Fungsi `plot_forecast`:
- Menyesuaikan index forecast agar dimulai tepat satu bulan setelah data aktual terakhir.
- Menambahkan nilai aktual terakhir sebagai titik awal agar garis forecast tersambung dengan garis data aktual.
- Menampilkan data aktual dan forecast RNN serta LSTM pada rentang Januari 2018 sampai Desember 2020.

In [ ]:
# Fungsi untuk plot forecast ke depan
def plot_forecast(sales_data, forecast_rnn_data, forecast_lstm_data):
    # Mendapatkan nilai dan index terakhir dari sales_data
    last_actual_value = sales_data['Sales'].iloc[-1]
    last_actual_date = sales_data.index[-1]

    # Menyusun prediksi agar dimulai tepat setelah data actual
    forecast_rnn_data.index = pd.date_range(start=last_actual_date + pd.DateOffset(months=1), periods=len(forecast_rnn_data), freq='MS')
    forecast_lstm_data.index = pd.date_range(start=last_actual_date + pd.DateOffset(months=1), periods=len(forecast_lstm_data), freq='MS')

    # Menambahkan nilai terakhir data actual sebagai titik awal prediksi
    forecast_rnn_data = pd.concat([pd.DataFrame({'RNN_Forecast': [last_actual_value]}, index=[last_actual_date]), forecast_rnn_data])
    forecast_lstm_data = pd.concat([pd.DataFrame({'LSTM_Forecast': [last_actual_value]}, index=[last_actual_date]), forecast_lstm_data])

    # Plot actual sales dan prediksi forecast
    plt.figure(figsize=(14, 8))
    plt.plot(sales_data.index, sales_data['Sales'], label='Actual Sales', color='blue', linewidth=2)
    plt.plot(forecast_rnn_data.index, forecast_rnn_data['RNN_Forecast'],
             label='RNN Forecast', color='orange', linestyle='--', linewidth=2)
    plt.plot(forecast_lstm_data.index, forecast_lstm_data['LSTM_Forecast'],
             label='LSTM Forecast', color='green', linestyle='--', linewidth=2)

    # Set the limits for x-axis to control the date range
    plt.xlim(pd.to_datetime('2018-01-01'), pd.to_datetime('2020-12-01'))

    # Formatting the x-axis to show dates nicely
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))  # Show every 3 months
    plt.gcf().autofmt_xdate()

    plt.title('Sales Forecast using RNN and LSTM', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

plot_forecast(sales_data, forecast_rnn_data, forecast_lstm_data)

## 11. Kesimpulan
1. Dataset penjualan ritel memiliki **trend naik** dan **seasonality tahunan** yang kuat dengan puncak setiap Desember. Karena itu, panjang sekuens 12 bulan cocok sebagai input.
2. **SimpleRNN** memiliki parameter lebih sedikit (±10 ribu) dan dilatih lebih cepat, tetapi rentan terhadap *vanishing gradient* untuk ketergantungan jangka panjang.
3. **LSTM** memiliki parameter sekitar 4 kali lebih banyak (±40 ribu) karena memakai *forget*, *input*, dan *output gate* serta *cell state*. Hal ini membuatnya lebih mampu menyimpan memori jangka panjang. `Dropout` ditambahkan untuk mengurangi overfitting.
4. Perbandingan performa dapat dilihat dari tabel **MAE/RMSE/MAPE** dan grafik *Actual vs Predictions* di atas. Model dengan error lebih kecil lebih baik dalam mengikuti pola penjualan. Karena bobot awal dan pelatihan bersifat acak, hasilnya dapat sedikit berbeda setiap kali notebook dijalankan.
5. Prediksi jangka panjang dilakukan secara rekursif, sehingga error dapat terakumulasi semakin jauh ke depan. Forecast 12 bulan sebaiknya dibaca sebagai gambaran pola, bukan angka pasti.

**Referensi:** Materi *RNN & LSTM* — Tegar Arifin Prasetyo, S.Si., M.Si; Hochreiter & Schmidhuber (1997), *Long Short-Term Memory*; TensorFlow Documentation.